# Diagnosis Inference Using Medication

## Architectural Flow

![Architectural Flow](architecture_flow.png)

# Diagnosis Inference Using Medication

This notebook infers a **more specific diagnosis** from a base diagnosis plus a list of medications.
Using the IMO Health Knowledge Graph `domainNarrowerByMedications` field, we iteratively drill
down the problem hierarchy — at each level the KG returns only narrower concepts that have
ties to the supplied medications — until we reach a leaf node (most specific diagnosis).

## Tool chain used in `dx-infer` mode
```python
DX_INFER_TOOLS = [
    normalize_medical_term,              # Normalize diagnosis & meds
    get_medication_diagnosis_proto,      # Iterative drill-down
]
```

## Key implementation details
- Base diagnosis is normalized as `domain="Problem"` to get its `default_lexical_code`
- Each medication is normalized as `domain="Medication"` to get medication codes
- `get_medication_diagnosis_proto` calls the KG `domainNarrowerByMedications` field
- Iteration continues until `numberOfDomainChildren == 0` (leaf reached) or empty response
- Results are sorted: non-leaf concepts first (by descending children count)

## Workflow

| Step | Action | API |
|------|--------|-----|
| 1 | Normalize base diagnosis (domain=Problem) | IMO Normalize |
| 2 | Normalize each medication (domain=Medication) | IMO Normalize |
| 3 | Call `domainNarrowerByMedications` iteratively | IMO KG GraphQL |
| 4 | Pick best candidate per level using clinical context | — |
| 5 | Stop at leaf (`numberOfDomainChildren == 0`) | — |

## 0. Setup — paths, imports, credentials

In [ ]:
import sys
import os

# Ensure the notebook's own directory is on the path so local
# kg_config.py and kg_api_client.py are imported directly.
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

import json
from IPython.display import display, Markdown

# ── Credential check ────────────────────────────────────────────────────────
# kg_config.py reads credentials from AWS SSM or environment variables.
# If SSM is unavailable locally, uncomment and set the env vars below.
#
# os.environ["IMO_NORMALIZE_CLIENT_ID"] = "..."
# os.environ["IMO_NORMALIZE_SECRET"]    = "..."
# os.environ["IMO_KG_CLIENT_ID"]        = "..."
# os.environ["IMO_KG_CLIENT_SECRET"]    = "..."

import kg_config as kc

missing = []
if not kc.IMO_NORMALIZE_CLIENT_ID:     missing.append("IMO_NORMALIZE_CLIENT_ID")
if not kc.IMO_NORMALIZE_CLIENT_SECRET: missing.append("IMO_NORMALIZE_SECRET")
if not kc.IMO_KG_CLIENT_ID:           missing.append("IMO_KG_CLIENT_ID")
if not kc.IMO_KG_CLIENT_SECRET:       missing.append("IMO_KG_CLIENT_SECRET")

if missing:
    print(f"\u26a0  Missing credentials: {missing}")
    print("   Set them as env vars (see comment above) or ensure AWS SSM access.")
else:
    print("\u2713  Credentials loaded")
    print(f"   Normalize URL : {kc.IMO_NORMALIZE_URL}")
    print(f"   KG GraphQL URL: {kc.KG_GRAPHQL_URL}")

## 1. Clinical Scenario

A 55-year-old male with a history of poorly controlled blood sugar, presenting with
polyuria, polydipsia, and peripheral neuropathy. Labs show HbA1c of 9.2%.

**Base diagnosis:** Type 2 diabetes mellitus

**Current medications:**
- Metformin 1000 mg BID
- Insulin glargine 30 units daily
- Empagliflozin 25 mg daily

In [ ]:
BASE_DIAGNOSIS = "type 2 diabetes mellitus"

MEDICATIONS = [
    "Metformin",
    "Insulin glargine",
    "Empagliflozin",
]

print(f"Base Diagnosis: {BASE_DIAGNOSIS}")
print(f"Medications   : {MEDICATIONS}")

## 2. Initialize KGApiClient

In [ ]:
from kg_api_client import KGApiClient

client = KGApiClient()
print("KGApiClient ready")

## Step 1 — Normalize the Base Diagnosis

Calls `normalize_medical_term(input_term, domain="Problem")` for the base diagnosis.
Returns a `default_lexical_code` — the stable IMO identifier used for KG lookups.

In [ ]:
print(f"Normalizing base diagnosis (domain=Problem): '{BASE_DIAGNOSIS}'...\n")

dx_result = client.normalize_medical_term(BASE_DIAGNOSIS, "Problem")
diagnosis_code = None
diagnosis_title = None

if dx_result.get("success") and dx_result.get("results"):
    matches = dx_result["results"][0].get("matches", [])
    if matches:
        best = matches[0]
        diagnosis_code = best["default_lexical_code"]
        diagnosis_title = best["title"]
        print(f"  \u2713  '{BASE_DIAGNOSIS}'")
        print(f"       code  = {diagnosis_code}")
        print(f"       title = '{diagnosis_title}'")
        print(f"       score = {best['score']}")
        if best.get("icd10_codes"):
            print(f"       ICD-10: {best['icd10_codes']}")
    else:
        print(f"  \u2717  No matches returned for '{BASE_DIAGNOSIS}'")
else:
    print(f"  \u2717  API error: {dx_result.get('error')}")

## Step 2 — Normalize Each Medication

Calls `normalize_medical_term(input_term, domain="Medication")` for each medication.
Collects `default_lexical_code` values into a list for the drill-down step.

In [ ]:
medication_codes = []  # list of codes for get_medication_diagnosis_proto
med_info = {}  # med name -> {code, title, score}

print("Normalizing medications (domain=Medication)...\n")
for med in MEDICATIONS:
    result = client.normalize_medical_term(med, "Medication")
    if result.get("success") and result.get("results"):
        matches = result["results"][0].get("matches", [])
        if matches:
            best = matches[0]
            code = best["default_lexical_code"]
            medication_codes.append(code)
            med_info[med] = {
                "code": code,
                "title": best["title"],
                "score": best["score"],
            }
            print(f"  \u2713  {med:<25}  code={code:<10}  title='{best['title']}'")
        else:
            print(f"  \u2717  {med:<25}  no matches returned")
    else:
        print(f"  \u2717  {med:<25}  API error: {result.get('error')}")

print(f"\n\u2713  Normalized {len(medication_codes)}/{len(MEDICATIONS)} medications")
print(f"   Medication codes: {medication_codes}")

## Step 3 — Iterative Drill-Down via `domainNarrowerByMedications`

Starting from the base diagnosis, we call `get_medication_diagnosis_proto` which queries
the KG for narrower problem concepts linked to our medications. We iterate:

1. Call the KG with current diagnosis code + medication codes
2. Pick the best candidate from the returned list (non-leaves first)
3. If `numberOfDomainChildren > 0` — drill deeper with that candidate's code
4. If `numberOfDomainChildren == 0` or empty response — stop (leaf reached)

Maximum 8 iterations to prevent infinite loops.

In [ ]:
MAX_ITERATIONS = 8

refinement_path = []  # list of {depth, code, title, children, chosen_reason, candidates}

# Starting point
refinement_path.append({
    "depth": 0,
    "code": diagnosis_code,
    "title": diagnosis_title,
    "children": "—",
    "chosen_reason": "starting concept",
    "candidates": [],
})

current_code = diagnosis_code
print(f"Starting drill-down from: '{diagnosis_title}' (code={diagnosis_code})")
print(f"Medication codes: {medication_codes}")
print(f"{'='*70}\n")

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f"--- Iteration {iteration}: querying domainNarrowerByMedications(code={current_code}) ---")
    
    result = client.get_medication_diagnosis_proto(current_code, medication_codes)
    
    if not result.get("success") or not result.get("lexical"):
        print(f"  \u2717  API error or null response: {result.get('error', 'no lexical data')}")
        print(f"  Stopping — using last valid concept as final diagnosis.")
        break
    
    items = result["lexical"].get("domainNarrowerByMedications", [])
    
    if not items:
        print(f"  Empty response — no narrower concepts linked to these medications.")
        print(f"  Stopping — current concept is the most specific available.")
        break
    
    # Sort: non-leaves first (by descending children count), then leaves
    items.sort(key=lambda x: -(x.get("numberOfDomainChildren") or 0))
    
    print(f"  Returned {len(items)} candidates:")
    for i, item in enumerate(items[:10]):  # show first 10
        children = item.get("numberOfDomainChildren", 0) or 0
        leaf_marker = " [LEAF]" if children == 0 else f" [{children} children]"
        print(f"    {i+1}. '{item['title']}' (code={item['code']}){leaf_marker}")
    if len(items) > 10:
        print(f"    ... and {len(items) - 10} more")
    
    # Pick the best candidate — prefer non-leaves for further drilling
    # In a real agent, clinical note context guides the pick
    picked = items[0]
    picked_children = picked.get("numberOfDomainChildren", 0) or 0
    reason = "most drillable (highest numberOfDomainChildren)" if picked_children > 0 else "leaf concept (no further children)"
    
    print(f"\n  \u2713 Picked: '{picked['title']}' (code={picked['code']}, children={picked_children})")
    print(f"    Reason: {reason}\n")
    
    refinement_path.append({
        "depth": iteration,
        "code": picked["code"],
        "title": picked["title"],
        "children": picked_children,
        "chosen_reason": reason,
        "candidates": [{"code": x["code"], "title": x["title"], "children": x.get("numberOfDomainChildren", 0)} for x in items[:10]],
    })
    
    if picked_children == 0:
        print(f"  Leaf reached — stopping iteration.")
        break
    
    current_code = picked["code"]

print(f"\n{'='*70}")
print(f"Drill-down complete after {len(refinement_path)-1} iteration(s).")

## Step 4 — Inferred Diagnosis Result

The final concept in our refinement path is the most specific diagnosis
that the KG can link to our medications.

In [ ]:
final = refinement_path[-1]

lines = [
    "## Diagnosis Inference Result",
    "",
    f"**Base Diagnosis:** {diagnosis_title} (`{diagnosis_code}`)",
    f"**Medications:** {', '.join(MEDICATIONS)}",
    "",
    f"**Inferred Specific Diagnosis:** {final['title']} (`{final['code']}`)",
    "",
    "### Refinement Path",
    "",
    "| Depth | Code | Title | Children Below | Chosen? |",
    "|-------|------|-------|----------------|----------|",
]

for step in refinement_path:
    marker = "\u2713" if step == final and step["depth"] > 0 else "\u2713" if step["depth"] == 0 else "\u2713"
    lines.append(
        f"| {step['depth']} | {step['code']} | {step['title']} | {step['children']} | {marker} {step['chosen_reason']} |"
    )

display(Markdown("\n".join(lines)))

## Step 5 — Candidates at Each Level

For transparency, here are all candidates returned by the KG at each drill-down level.

In [ ]:
for step in refinement_path:
    if step["depth"] == 0:
        continue
    print(f"{'\u2500'*60}")
    print(f"  Depth {step['depth']} — drilling into: (parent code fed to API)")
    print(f"  Picked: '{step['title']}' (code={step['code']})")
    print(f"  Reason: {step['chosen_reason']}")
    if step["candidates"]:
        print(f"  All candidates at this level:")
        for i, cand in enumerate(step["candidates"]):
            picked_mark = " \u25c0 PICKED" if cand["code"] == step["code"] else ""
            children = cand.get("children", 0) or 0
            print(f"    {i+1}. '{cand['title']}' (code={cand['code']}, children={children}){picked_mark}")
    print()

## Appendix — Additional Clinical Scenarios

Try these alternative scenarios by changing `BASE_DIAGNOSIS` and `MEDICATIONS` above:

| Scenario | Base Diagnosis | Medications |
|----------|---------------|-------------|
| Tonsilitis + Amoxicillin | tonsilitis | Amoxicillin |
| Pharyngitis + Penicillin V | pharyngitis | Penicillin V |
| Otitis media + Amoxicillin | otitis media | Amoxicillin |
| Heart failure + GDMT | heart failure | Sacubitril/valsartan, Carvedilol, Spironolactone, Dapagliflozin |